## 1. Architecture reminders and intuition

**Q1 — Sequence attention vs. graph attention.**
In sequence Transformers, self-attention generally allows every token to attend to every other token in the sequence.

In graph models, attention can be restricted to neighboring nodes connected through graph edges. Therefore, attention operates according to the graph topology rather than a predefined sequence order.

**Q2 — Why sequence positional encodings do not transfer to graphs.**
Sequences have a natural ordering of elements, whereas graphs do not have a fixed ordering of nodes.

The node arrangement in a graph can be arbitrary. Therefore, sequence-based positional encodings cannot directly represent the structural position of nodes in arbitrary graphs.

Graph models require structural positional encodings that capture information about graph connectivity and topology.

**Q3 — Two advantages of attention over graph neighbourhoods.**
1. **Adaptive importance:** Attention assigns different weights to neighboring nodes according to their relevance instead of treating all neighbors equally.

2. **Selective information aggregation:** The model can focus on more informative neighbors and reduce the influence of less relevant neighboring nodes.

**Q4 — `TransformerConv` vs. dense all-pairs self-attention.**
PyG's TransformerConv calculates attention over the graph edges supplied through `edge_index`.

It performs local attention between connected nodes rather than computing attention between every possible pair of nodes.

In contrast, dense all-pairs self-attention considers every node pair, regardless of whether an edge connects them.


**Q5 — What a structural PE adds.**
Structural positional encoding provides additional information about the position and structural role of a node within the graph.

It captures information derived from graph topology and complements the original node features, helping the model learn more informative node representations.


## Exercise 2: Implement a Graph Transformer

### Objective

Implement a two-layer Graph Transformer using PyTorch Geometric's TransformerConv layer for node classification.

The architecture consists of:
- First TransformerConv layer with 32 channels per head and 2 attention heads.
- ReLU activation.
- Second TransformerConv layer with 1 attention head.
- Output dimension equal to the number of node classes.


In [13]:
# Import required libraries

import torch
import torch.nn.functional as F

from torch_geometric.datasets import Planetoid
from torch_geometric.nn import TransformerConv

import matplotlib.pyplot as plt
import numpy as np

print("PyTorch version:", torch.__version__)
print("PyTorch Geometric imported successfully!")

PyTorch version: 2.15.0+cpu
PyTorch Geometric imported successfully!



### 2.2 Dataset Loading

We use the Cora dataset for node classification.

Cora is a citation network in which:
- Nodes represent scientific publications.
- Edges represent citation relationships.
- Node features represent publication characteristics.
- Labels represent publication categories.

The objective is to predict the category of each publication based on its features and graph connections.


In [14]:
# Load Cora dataset
dataset = Planetoid(
    root='../data/Planetoid',
    name='Cora'
)

data = dataset[0]

print(data)

Data(x=[2708, 1433], edge_index=[2, 10556], y=[2708], train_mask=[2708], val_mask=[2708], test_mask=[2708])


In [15]:
print("Number of nodes:", data.num_nodes)
print("Number of edges:", data.num_edges)

print("Number of node features:", dataset.num_features)
print("Number of classes:", dataset.num_classes)

print("Node feature shape:", data.x.shape)
print("Edge index shape:", data.edge_index.shape)
print("Labels shape:", data.y.shape)

Number of nodes: 2708
Number of edges: 10556
Number of node features: 1433
Number of classes: 7
Node feature shape: torch.Size([2708, 1433])
Edge index shape: torch.Size([2, 10556])
Labels shape: torch.Size([2708])



### 2.4 Graph Transformer Implementation

We implement a Graph Transformer using two TransformerConv layers.

The first layer uses two attention heads, each producing 32 channels. The outputs are concatenated, resulting in 64 features per node.

A ReLU activation is applied between the two layers.

The second layer uses one attention head and produces one score for each class.

The model returns raw class scores (logits), without applying Softmax.


In [16]:
class GraphTransformer(torch.nn.Module):

    def __init__(self, in_channels, num_classes):

        super().__init__()

        # First TransformerConv layer
        self.conv1 = TransformerConv(
            in_channels=in_channels,
            out_channels=32,
            heads=2,
            concat=True
        )

        # Second TransformerConv layer
        self.conv2 = TransformerConv(
            in_channels=64,
            out_channels=num_classes,
            heads=1,
            concat=False
        )

    def forward(self, x, edge_index):

        # First graph transformer layer
        x = self.conv1(x, edge_index)

        # ReLU activation
        x = F.relu(x)

        # Second graph transformer layer
        x = self.conv2(x, edge_index)

        return x

In [17]:
# Initialize Graph Transformer

model = GraphTransformer(
    in_channels=dataset.num_features,
    num_classes=dataset.num_classes
)

print(model)

GraphTransformer(
  (conv1): TransformerConv(1433, 32, heads=2)
  (conv2): TransformerConv(64, 7, heads=1)
)


In [18]:
total_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print("Total trainable parameters:", total_params)

Total trainable parameters: 368924



### 2.6 Model Output Verification

We pass the node features and graph connectivity information through the Graph Transformer.

The model produces raw class scores for every node.

We verify that:
- The number of output rows equals the number of graph nodes.
- The number of output columns equals the number of classes.


In [19]:
# Forward pass

out = model(
    data.x,
    data.edge_index
)

print("Output shape:", out.shape)

print("Number of nodes:", data.num_nodes)
print("Number of classes:", dataset.num_classes)

Output shape: torch.Size([2708, 7])
Number of nodes: 2708
Number of classes: 7


In [20]:
assert out.shape[0] == data.num_nodes

assert out.shape[1] == dataset.num_classes

print("Model output dimensions verified successfully!")

Model output dimensions verified successfully!



### 2.7 TransformerConv Default Arguments

According to the PyTorch Geometric documentation:

- `concat=True`: Concatenates the outputs of multiple attention heads.
- `edge_dim=None`: No edge feature dimension is specified by default.
- `dropout=0.0`: No attention dropout is applied by default.

In our implementation, the first layer uses `concat=True` to combine the outputs of two attention heads.

The second layer uses `concat=False` because only one output representation per node is required, with a dimension equal to the number of classes.


In [21]:
print(data)
print(model)
print(out.shape)

Data(x=[2708, 1433], edge_index=[2, 10556], y=[2708], train_mask=[2708], val_mask=[2708], test_mask=[2708])
GraphTransformer(
  (conv1): TransformerConv(1433, 32, heads=2)
  (conv2): TransformerConv(64, 7, heads=1)
)
torch.Size([2708, 7])


In [22]:
torch.Size([2708, 7])

torch.Size([2708, 7])